In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

In [5]:
shared_path = "../Data/R3Data"

### 9. hormonal_mens
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [7]:
idx = 9
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

hormonal_mens
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'AGE_MENARCHE', 'AGE_FIRST_LIVE_BIRTH', 'AGE_MENOPAUSE', 'AGE_HYSTERECTOMY', 'AGE_RIGHT_OVARY_REMOVAL', 'AGE_LEFT_OVARY_REMOVAL', 'PARITY_COUNT', 'PREGNANCY_COUNT', 'LAST_MENSTRUAL_DATE', 'MENSTRUAL_STATUS_CD', 'EXAM_COMPLETED_DATE']


In [8]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['hormonal_mens.txt', 'hormonal_mens.csv', 'hormonal_mens.csv']

### Cancer

In [9]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [10]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [11]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,INTERNAL_EXAM_ID,PATIENT_ID,ENTRY_NAME,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,CYCLE_PHASE,PREGNANT_IND,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
0,4334469937,69423837,83749463,1152783,Hormonal Contraceptives,21,0,0,Y,N,12,25,0,0,0,0,2.0,2.0,03/27/2020 00:00:00,NaN,N,NaN,06/25/2020
1,4334469937,69423837,83749463,1152783,Raloxifene,0,0,0,N,Y,12,25,0,0,0,0,2.0,2.0,03/27/2020 00:00:00,NaN,N,NaN,06/25/2020
2,4334469937,69423837,83749463,1152783,Tamoxifen,0,0,0,N,Y,12,25,0,0,0,0,2.0,2.0,03/27/2020 00:00:00,NaN,N,NaN,06/25/2020
3,4334469937,69423837,83749463,1152783,Hormonal Contraceptives,21,31,0,N,N,12,25,0,0,0,0,2.0,2.0,03/27/2020 00:00:00,NaN,N,NaN,06/25/2020
4,4334469937,69423837,83749463,1152783,Progesterone,0,0,0,N,Y,12,25,0,0,0,0,2.0,2.0,03/27/2020 00:00:00,NaN,N,NaN,06/25/2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
944428,4334331716,450009554,73370350,1210201,Estrogen,0,0,0,N,Y,13,36,51,0,0,0,1.0,3.0,NaN,NaN,N,POSTNAT,12/21/2022
944429,4334331716,450009554,73370350,1210201,Tamoxifen,0,0,0,N,Y,13,36,51,0,0,0,1.0,3.0,NaN,NaN,N,POSTNAT,12/21/2022
944430,4334331716,450009554,73370350,1210201,Other hormones,0,0,0,N,Y,13,36,51,0,0,0,1.0,3.0,NaN,NaN,N,POSTNAT,12/21/2022
944431,4334331716,450009554,73370350,1210201,Arimidex,0,0,0,N,Y,13,36,51,0,0,0,1.0,3.0,NaN,NaN,N,POSTNAT,12/21/2022


In [12]:
final_df['EXAM_COMPLETED_DATE'] = pd.to_datetime(final_df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
final_df['EXAM_COMPLETED_DATE'] = final_df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')
final_df['LAST_MENSTRUAL_DATE'] = pd.to_datetime(final_df['LAST_MENSTRUAL_DATE'], format='%m/%d/%Y 00:00:00')
final_df['LAST_MENSTRUAL_DATE'] = final_df['LAST_MENSTRUAL_DATE'].dt.strftime('%Y-%m-%d')

In [13]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries <span style="color: red;"> on extracted cols only</span>, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [22]:
unique_df = final_df[extract_cols].copy()

In [23]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [24]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [25]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [26]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-07-26,7
1,4330018595,63737104,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-08-19,7
2,4330018595,60103700,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
3,4330018595,60695725,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
4,4330018595,60690108,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
5,4330018595,60667802,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-05,7
6,4330018595,60693047,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-05,7
7,4330018595,69461777,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-30,7
8,4330018595,69089910,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-07-28,7
9,4330018595,69089931,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-07-28,7


In [27]:
unique_df['duplicate_count'].unique()

array([ 7,  8,  1,  9,  4,  6, 10, 12, 11,  5,  3,  2, 15, 13, 14, 20, 16,
       18, 32, 17, 21, 28, 19, 30, 24])

In [28]:
unique_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-07-26,7
1,4330018595,63737104,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-08-19,7
2,4330018595,60103700,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
3,4330018595,60695725,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
4,4330018595,60690108,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119713,4339959661,68434457,13,0,0,0,0,0,0.0,0.0,2020-11-05,PRE,2020-11-09,8
119714,4339959661,454148687,13,0,0,0,0,0,0.0,0.0,2021-10-28,PRE,2021-11-10,9
119715,4339959661,454612462,13,0,0,0,0,0,0.0,0.0,2021-10-28,PRE,2021-11-15,9
119716,4339959661,453550821,13,0,0,0,0,0,0.0,0.0,2021-10-28,PRE,2021-11-22,9


In [29]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)

### Control

In [30]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['hormonal_mens_controls.txt', 'hormonal_mens_controls.csv']

In [31]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [32]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_36237/483784814.py:10: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  current_df = pd.read_csv(file_path, encoding='latin-1')


✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [33]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,INTERNAL_EXAM_ID,PATIENT_ID,ENTRY_NAME,AGE_FIRST_USE,AGE_LAST_USE,DURATION,CURRENT_USE_IND,NEVER_USE_IND,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,CYCLE_PHASE,PREGNANT_IND,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE
0,4334585594,68008794,81307232,45497467,Progesterone,0,0,0,N,Y,13,23,53,0,0,0,2.0,2.0,03/01/2001 00:00:00,NaN,N,POSTNAT,12/08/2020
1,4334585594,68008794,81307232,45497467,Hormonal Contraceptives,28,29,0,N,N,13,23,53,0,0,0,2.0,2.0,03/01/2001 00:00:00,NaN,N,POSTNAT,12/08/2020
2,4334585594,68008794,81307232,45497467,Tamoxifen,0,0,0,N,Y,13,23,53,0,0,0,2.0,2.0,03/01/2001 00:00:00,NaN,N,POSTNAT,12/08/2020
3,4334585594,68008794,81307232,45497467,Other hormones,0,0,0,N,Y,13,23,53,0,0,0,2.0,2.0,03/01/2001 00:00:00,NaN,N,POSTNAT,12/08/2020
4,4334585594,68008794,81307232,45497467,Arimidex,0,0,0,N,Y,13,23,53,0,0,0,2.0,2.0,03/01/2001 00:00:00,NaN,N,POSTNAT,12/08/2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2210186,4334115607,459587063,73754997,1040461,Arimidex,0,0,0,N,Y,11,0,46,46,46,46,0.0,0.0,NaN,NaN,N,POSTSUR,11/19/2022
2210187,4334115607,459587063,73754997,1040461,Tamoxifen,0,0,0,N,Y,11,0,46,46,46,46,0.0,0.0,NaN,NaN,N,POSTSUR,11/19/2022
2210188,4334115607,459587063,73754997,1040461,Hormonal Contraceptives,31,33,0,N,N,11,0,46,46,46,46,0.0,0.0,NaN,NaN,N,POSTSUR,11/19/2022
2210189,4334115607,459587063,73754997,1040461,Estrogen,45,49,0,N,N,11,0,46,46,46,46,0.0,0.0,NaN,NaN,N,POSTSUR,11/19/2022


In [34]:
final_df['EXAM_COMPLETED_DATE'] = pd.to_datetime(final_df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
final_df['EXAM_COMPLETED_DATE'] = final_df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')
final_df['LAST_MENSTRUAL_DATE'] = pd.to_datetime(final_df['LAST_MENSTRUAL_DATE'], format='%m/%d/%Y 00:00:00')
final_df['LAST_MENSTRUAL_DATE'] = final_df['LAST_MENSTRUAL_DATE'].dt.strftime('%Y-%m-%d')

In [35]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries <span style="color: red;"> on extracted cols only</span>, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [36]:
unique_df = final_df[extract_cols].copy()

In [37]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [38]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [39]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [40]:
unique_df['duplicate_count'].unique()

array([ 7,  2,  6,  1,  8,  9,  3, 10,  4, 11, 14,  5, 12, 16, 13, 15, 20,
       18, 17, 28, 22, 34, 24, 19])

In [41]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,duplicate_count
0,4330000534,78048053,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2018-03-28,7
1,4330000534,64787403,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-05-18,7
2,4330000534,63433254,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-06-20,7
3,4330000534,63280481,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-06-20,7
4,4330000534,63280358,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-06-20,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292108,4339982047,450915296,13,21,45,0,0,0,2.0,2.0,NaN,POSTNAT,2022-10-12,7
292109,4339988199,65403287,11,34,53,0,0,0,1.0,1.0,NaN,POSTNAT,2018-11-28,7
292110,4339988199,61697072,11,34,53,0,0,0,1.0,1.0,NaN,POSTNAT,2020-02-21,7
292111,4339988199,67718778,11,34,53,0,0,0,1.0,1.0,NaN,POSTNAT,2021-02-25,7


In [42]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)